# A single-stage detector, from scratch

The YOLO idea in its simplest form: a grid, a fixed number of predictions per cell, and one loss combining classification with regression.

**Runs on:** GPU recommended — about 30 minutes on CPU &nbsp;·&nbsp; **Slides:** [Chapter 12 — Object Detection](../../../course-web-slides/ch12/index.html) &nbsp;·&nbsp; **Section:** 02 — Building a detector

---

## The central idea

Chapter 8's classifier answers *what is in this image*. A detector must answer *what, and where, and how many* — and the last part is what makes the output shape awkward, because it varies per image.

**YOLO's answer: fix the output shape.** Divide the image into a grid; each cell predicts a fixed number of boxes and a class distribution. The number of predictions is now constant, and the problem becomes an ordinary supervised one.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

IMG, GRID = 256, 8
CELL = IMG // GRID

rng = np.random.default_rng(1)
img = rng.random((IMG, IMG, 3)) * 0.25 + 0.6
box = np.array([70., 90., 190., 210.])
cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.imshow(img)
for k in range(1, GRID):
    ax.axhline(k * CELL, color="w", lw=.6)
    ax.axvline(k * CELL, color="w", lw=.6)
ax.add_patch(patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                               fill=False, edgecolor="#ff7a1a", lw=2.5))
gx, gy = int(cx // CELL), int(cy // CELL)
ax.add_patch(patches.Rectangle((gx*CELL, gy*CELL), CELL, CELL,
                               facecolor="#12b886", alpha=.35))
ax.plot(cx, cy, "o", c="#12b886", ms=9)
ax.set_title(f"The object's centre falls in cell ({gx}, {gy})\n"
             f"— that cell is responsible for predicting it")
ax.axis("off"); plt.show()

## The output tensor

In [ ]:
NUM_CLASSES = 3
BOXES_PER_CELL = 2

# Per cell: for each box, (x, y, w, h, objectness); plus one class distribution.
per_cell = BOXES_PER_CELL * 5 + NUM_CLASSES
print(f"output shape: ({GRID}, {GRID}, {per_cell})")
print(f"  {BOXES_PER_CELL} boxes x (x, y, w, h, objectness) = "
      f"{BOXES_PER_CELL * 5}")
print(f"  + {NUM_CLASSES} class scores")
print(f"total predictions per image: {GRID * GRID * BOXES_PER_CELL}")

**128 candidate boxes for every image, always** — most of them predicting *nothing here*. That imbalance is the central difficulty of single-stage detection and the thing the loss has to handle.

## Encoding a target

In [ ]:
def encode(boxes, classes, grid=GRID, img_size=IMG,
           num_classes=NUM_CLASSES):
    target = np.zeros((grid, grid, 5 + num_classes), dtype="float32")
    cell = img_size / grid
    for b, c in zip(boxes, classes):
        cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
        gx, gy = int(cx // cell), int(cy // cell)
        target[gy, gx, 0] = (cx - gx * cell) / cell     # offset WITHIN the cell
        target[gy, gx, 1] = (cy - gy * cell) / cell
        target[gy, gx, 2] = (b[2] - b[0]) / img_size    # size relative to image
        target[gy, gx, 3] = (b[3] - b[1]) / img_size
        target[gy, gx, 4] = 1.0                          # objectness
        target[gy, gx, 5 + c] = 1.0
    return target

t = encode([box], [1])
print("cells containing an object:", int(t[..., 4].sum()), "of", GRID * GRID)
print("that cell's vector:", t[gy, gx].round(3))

Two different normalizations, and mixing them is a classic bug:
- **Centre** is an offset *within its cell*, in [0, 1].
- **Size** is relative to the *whole image*, in [0, 1].

The first keeps the regression local and easy; the second lets one cell predict a box larger than itself.

## The model

In [ ]:
import keras
from keras import layers

def detector(grid=GRID, boxes_per_cell=BOXES_PER_CELL,
             num_classes=NUM_CLASSES, img_size=IMG):
    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = layers.Rescaling(1./255)(inputs)
    for f in [32, 64, 128, 256, 512]:
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D(2)(x)
    # 256 -> 8 after five halvings; the spatial grid IS the output grid.
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    outputs = layers.Conv2D(boxes_per_cell * 5 + num_classes, 1)(x)
    return keras.Model(inputs, outputs)

model = detector()
print("output shape:", model.output.shape)
print(f"{model.count_params():,} parameters")

**The backbone's spatial grid is the output grid.** No `Flatten`, no `Dense` — the last layer is a 1×1 convolution producing the per-cell vector. Everything from chapters 8 and 9 applies unchanged.

## The loss: three terms, deliberately unequal

In [ ]:
from keras import ops

LAMBDA_COORD = 5.0     # boxes matter more than the many empty cells
LAMBDA_NOOBJ = 0.5     # ...and empty cells must not drown everything

def detection_loss(y_true, y_pred, num_classes=NUM_CLASSES):
    obj = y_true[..., 4:5]                       # 1 where an object is
    noobj = 1.0 - obj

    xy_loss = ops.sum(obj * ops.square(y_true[..., 0:2] - y_pred[..., 0:2]))
    wh_loss = ops.sum(obj * ops.square(
        ops.sqrt(ops.maximum(y_true[..., 2:4], 1e-6))
        - ops.sqrt(ops.maximum(y_pred[..., 2:4], 1e-6))))
    obj_loss = ops.sum(obj * ops.square(y_true[..., 4:5] - y_pred[..., 4:5]))
    noobj_loss = ops.sum(noobj * ops.square(y_true[..., 4:5] - y_pred[..., 4:5]))
    cls_loss = ops.sum(obj * ops.square(
        y_true[..., 5:5+num_classes] - y_pred[..., 5:5+num_classes]))

    return (LAMBDA_COORD * (xy_loss + wh_loss)
            + obj_loss + LAMBDA_NOOBJ * noobj_loss + cls_loss)

Three details that are all responses to the same imbalance:

**`obj` masks almost every term.** Coordinates are only penalised where there is an object to have coordinates.

**`LAMBDA_NOOBJ = 0.5`.** With 126 empty cells against 2 full ones, the objectness term would otherwise be dominated by cells learning to say *nothing here*, and the model would learn nothing else.

**The square root on width and height.** A 10-pixel error on a 20-pixel box matters far more than on a 200-pixel box; the square root compresses the large end so both are penalised comparably.

## Decoding predictions back to boxes

In [ ]:
def decode(pred, grid=GRID, img_size=IMG, num_classes=NUM_CLASSES,
           threshold=0.5):
    cell = img_size / grid
    boxes, scores, classes = [], [], []
    for gy in range(grid):
        for gx in range(grid):
            v = pred[gy, gx]
            if v[4] < threshold:
                continue
            cx = (gx + v[0]) * cell
            cy = (gy + v[1]) * cell
            w, h = v[2] * img_size, v[3] * img_size
            boxes.append([cx - w/2, cy - h/2, cx + w/2, cy + h/2])
            scores.append(float(v[4]))
            classes.append(int(np.argmax(v[5:5+num_classes])))
    return np.array(boxes), np.array(scores), np.array(classes)

b, s, c = decode(t)          # decode the target we encoded above
print("recovered:", b.round(1), "class", c, "score", s)
print("original: ", box)

Encode then decode should return what you started with. **Test that round-trip before training anything** — an encoding bug is invisible in the loss curve and fatal in the output.

## What this simple version does not have

| Missing | Why real detectors add it |
|---|---|
| **Anchor boxes** | Objects have characteristic shapes; predicting an offset from a prior beats predicting from scratch. |
| **Multi-scale features** | One grid means one object size. FPNs detect at several resolutions. |
| **Focal loss** | A principled replacement for `LAMBDA_NOOBJ` — downweights easy negatives continuously. |
| **IoU-based losses** | GIoU and CIoU optimise the metric you actually report, rather than a proxy. |

Notebook 03 uses a pretrained detector that has all four.

---

## What to take away

- A grid with a fixed number of predictions per cell turns a variable-length output into a fixed one.
- Centre offsets are per-cell; sizes are per-image. Do not mix them.
- The loss must downweight empty cells or it learns only to say *nothing here*.
- Test the encode/decode round-trip before training — that bug is invisible in the loss.